## 7. Deployment of specialized LLMs to use with Cypher (5 LLMs in total)

- [7a. Cypher Generation: The Good, The Bad, and The Messy](https://towardsdatascience.com/cypher-generation-the-good-the-bad-and-the-messy-4ec119dd72ea)
    - This is not really a specialized LLM, but instructions on how to prepare a dataset for fine-tuning.

- [7b. Llama3 Text2Cypher Demo](https://huggingface.co/collections/tomasonjo/llama3-text2cypher-demo-6647a9eae51e5310c9cfddcf)
    - There are 5 models derived from the Llama3 model (with different sizes).

- [7c. Text2Cypher GitHub Repository](https://github.com/neo4j-labs/text2cypher)
    - A description of the approach in general.

- [7d. Text2Cypher Demo 16-bit](https://huggingface.co/tomasonjo/text2cypher-demo-16bit)
    - A specific 16-bit model.

- [7e. Text2Cypher Demo 6-bit GGUF](https://huggingface.co/tomasonjo/text2cypher-demo-6bit-gguf)
    - A specific 6-bit model.

Ignoring option 7a, we will evaluate 5 different fine-tuned llama3 variants:
- https://huggingface.co/tomasonjo/text2cypher-demo-16bit
- https://huggingface.co/tomasonjo/text2cypher-demo-16bit-gguf
- https://huggingface.co/tomasonjo/text2cypher-demo-4bit-gguf
- https://huggingface.co/tomasonjo/text2cypher-demo-8bit-gguf
- https://huggingface.co/tomasonjo/text2cypher-demo-6bit-gguf

These are essentially the same model based on unsloth/llama-3-8b-Instruct, fine-tuned to demo-16bit and then quantized.

We will try using ollama module with gguf files

# Ollama installation

Ollama could be installed from https://ollama.com/download. 

The gguf models should be downloaded from huggingface, and initialized with ollama. First, a Modelfile should be created, with a single line

```text
FROM <FILENAME>
```

where filename is path to gguf file (e.g. text2cypher-demo-4bit-gguf-unsloth.Q4_K_M.gguf).

Then, ollama model should be created from Modelfile:

```bash
ollama create MODELNAME -f Modelfile
```

where MODELNAME is the name of the model (e.g. text2cypher-4bit)

The model then can be accesses with ollama library. 

We will create these models:

- text2cypher-4bit
- text2cypher-6bit
- text2cypher-8bit
- text2cypher-16bit

In [2]:
# Testing the model

test_schema = """Node properties: - **Question** - `favorites`: INTEGER Example: "0" - `answered`: BOOLEAN - `text`: STRING Example: "### This is: Bug ### Specifications OS: Win10" - `link`: STRING Example: "https://stackoverflow.com/questions/62224586/playg" - `createdAt`: DATE_TIME Min: 2020-06-05T16:57:19Z, Max: 2020-06-05T21:49:16Z - `title`: STRING Example: "Playground is not loading with apollo-server-lambd" - `id`: INTEGER Min: 62220505, Max: 62224586 - `upVotes`: INTEGER Example: "0" - `score`: INTEGER Example: "-1" - `downVotes`: INTEGER Example: "1" - **Tag** - `name`: STRING Example: "aws-lambda" - **User** - `image`: STRING Example: "https://lh3.googleusercontent.com/-NcFYSuXU0nk/AAA" - `link`: STRING Example: "https://stackoverflow.com/users/10251021/alexandre" - `id`: INTEGER Min: 751, Max: 13681006 - `reputation`: INTEGER Min: 1, Max: 420137 - `display_name`: STRING Example: "Alexandre Le" Relationship properties: The relationships: (:Question)-[:TAGGED]->(:Tag) (:User)-[:ASKED]->(:Question)"""
question = "Identify the top 5 questions with the most downVotes."

test_messages = [
      {"role": "system", "content": "Given an input question, convert it to a Cypher query. No pre-amble."},
      {"role": "user", "content": f"""Based on the Neo4j graph schema below, write a Cypher query that would answer the user's question:
{test_schema}

Question: {question}
Cypher query:"""}
]

import ollama

response = ollama.chat(model='text2cypher-4bit', messages=test_messages)

print(response['message']['content'])


MATCH (q:Question)
RETURN q.title, q.downVotes
ORDER BY q.downVotes DESC
LIMIT 5 


## Loading the actual schema and doing actual tests

In [3]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


In [4]:
from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)


In [5]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [69]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = [llm_output]
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results


## Questions and schema

In [7]:
questions = [
    "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)",
    "What is the evidence linking TDP-43 to cancer in animal models?",
    "What (or is there) is the clinical evidence linking BRAF to Melanoma?"
]

In [36]:
# Graph schema
# pip install langchain_community neo4j

from langchain_community.graphs import Neo4jGraph

# Enhanced schema is too big to fit into context window of the model, so we will be using short schema:


short_schema = """
Node properties:
- **Disease**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
- **Association**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **Literature.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **AnimalModel.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 11
  - `score`: FLOAT Example: "0.4947"
- **RnaExpression.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.04389890211572866"
- **Gene**
  - `targetInModel`: STRING Example: "Eya1"
  - `targetInModelMgiId`: STRING Example: "MGI:109344"
  - `targetFromSourceId`: STRING Example: "ENSG00000104313"
- **KnownDrug.GeneToDiseaseAssociation**
  - `score`: FLOAT Example: "0.1"
  - `source`: STRING Example: "chembl"
- **SomaticMutation.GeneToDiseaseAssociation**
  - `score`: FLOAT Example: "0.25"
- **AffectedPathway.GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 21
  - `score`: FLOAT Example: "1.0"
- **GeneticAssociation.GeneToDiseaseAssociation**
  - `score`: FLOAT Example: "0.35"
Relationship properties:

The relationships:
(:Disease)-[:IS_PART_OF]->(:GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:AnimalModel.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:Literature.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:GeneticAssociation.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:SomaticMutation.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:KnownDrug.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:AffectedPathway.GeneToDiseaseAssociation)
(:Disease)-[:IS_PART_OF]->(:RnaExpression.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:RnaExpression.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:SomaticMutation.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:GeneticAssociation.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:Literature.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:AffectedPathway.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:AnimalModel.GeneToDiseaseAssociation)
(:Gene)-[:IS_PART_OF]->(:KnownDrug.GeneToDiseaseAssociation)
"""


In [37]:
from langchain_core.prompts import PromptTemplate

system_prompt_generic = """
You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a Neo4j database with biological data that accepts queries cypher queries. Scientists will provide you questions, and you should output correct cypher statements that will generate results. 

{schema}

"""

enhanced_schema_description = f"""\
This is graph schema:
--------------------------------------------
{short_schema}
--------------------------------------------    
"""

system_prompt_enhanced_schema = system_prompt_generic.format(schema = enhanced_schema_description)

print(system_prompt_enhanced_schema)



You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a Neo4j database with biological data that accepts queries cypher queries. Scientists will provide you questions, and you should output correct cypher statements that will generate results. 

This is graph schema:
--------------------------------------------

Node properties:
- **Disease**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
- **Association**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
- **GeneToDiseaseAssociation**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Exam

In [38]:
def generate_messages_simple(question):
    schema = short_schema

    messages = [
        {"role": "system", "content": "Given an input question, convert it to a Cypher query. No pre-amble."},
        {"role": "user", "content": f"""Based on the Neo4j graph schema below, write a Cypher query that would answer the user's question:
{schema}

Question: {question}
Cypher query:"""}
    ]
    return messages


def generate_messages(question):

    messages = [
        {"role": "system", "content": system_prompt_enhanced_schema},
        {"role": "user", "content": f"""
Given the graph schema above, write a Cypher query that would answer the following question: 
{question}
"""}
    ]
    return messages


## Model initialization and query


In [51]:
def run_ollama(model, messages):
    response = ollama.chat(model=model, messages=messages)
    return response['message']['content']


def run_llm_7(model, question):
    try:
        messages = generate_messages(question)
        response = run_with_timeout(run_ollama, 90, model, messages)
        return response
    except Exception as e:
        print(e)
        return None


## Running all quantized models

In [64]:
models = ["text2cypher-16bit", "text2cypher-8bit", "text2cypher-6bit", "text2cypher-4bit"]
niter = 10

todo = [(m, q) for m in models for q in questions for _ in range(niter)]

llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_7(llm_model, question))



Prompting LLM: 100%|██████████| 120/120 [03:29<00:00,  1.75s/it]


In [71]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [01:01<00:00,  1.95it/s]


In [ ]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("07-evaluations.xlsx", index=False)
with open("07-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

[[{'query': 'MATCH (g:Gene {targetInModelMgiId: "MGI:1856307"})-[:IS_PART_OF]->(a:Association)\nWHERE a.literature CONTAINS "amyotrophic lateral sclerosis" AND a.score > 0.1\nRETURN g.targetInModel AS Gene, a.literature AS Evidence, a.score AS Strength\nORDER BY a.score DESC\nLIMIT 5',
   'success': True,
   'result': [],
   'time': 0.11470913887023926}],
 [{'query': 'Cypher query:\nMATCH (d:Disease {name: "amyotrophic lateral sclerosis (ALS)"})<-[:IS_PART_OF]-(a:Association)\nWHERE EXISTS {\n  MATCH (g:Gene {targetFromSourceId: "ENSG00000148148"})-[:IS_PART_OF]->(a)\n}\nRETURN "There is evidence between TDP-43 and ALS" AS statement, a.score AS association_score\nOPTIONAL MATCH (d)<-[:IS_PART_OF]-(g:Gene)-[:IS_PART_OF]->(a2:Association)\nRETURN "The strongest association score found is: " + toString(a2.score) AS strong_association\nUNION ALL\nRETURN "No specific associations were found." AS statement',
   'success': False,
   'exception': '[Statement.SyntaxError] Invalid input \'query\